In [ ]:
from pathlib import Path
import os
import sys

for repo_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = repo_root / "src"
    if (src_dir / "adaptive_algorithms").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        cache_dir = repo_root / ".cache"
        cache_dir.mkdir(parents=True, exist_ok=True)
        os.environ.setdefault("XDG_CACHE_HOME", str(cache_dir))
        mpl_config_dir = cache_dir / "matplotlib"
        mpl_config_dir.mkdir(parents=True, exist_ok=True)
        os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))
        break
else:
    raise RuntimeError("Run this notebook from inside the adaptive-algorithms repo, or install the package in editable mode.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt 

from adaptive_algorithms import AdaptiveAlgorithm, LowRankEnergy, load_dataset

In [ ]:
X, labels = load_dataset("test", n_test=500)

# Usage 

With a dataset $X \in \mathbb{R}^{d \times n}$ (i.e., columns are the datapoints of interest), then we do the following to identify a prototype set:
* __Define an Energy object__: ``energy = LowRankEnergy(X, p=p)``
    * This handles all the updating of distances based on the choice of (i) the distance $d(x_i, \mathcal{S})$ and (ii) the value of power, $p$
* __Define a AdaptiveAlgorithm object__: ``algorithm = AdaptiveAlgorithm(energy)``
    * This object handles the interactivity of selecting which points to add to the prototype set. This object references the ``energy`` object previously defined. 

Prototype set selection then proceeds as follows
```python
# Build Phase
algorithm.build_phase(k, method="sampling")  # method is "sampling" or "search", selecting k prototypes

# Swap Phase
algorithm.swap_phase(method="search")     # method is "sampling" or "search"
```

## Adaptive Search Build

In [ ]:
energy = LowRankEnergy(X, p=2)
adapsearch = AdaptiveAlgorithm(energy, seed=10)
adapsearch.build_phase(5, "search")  # build prototype set of k = 5 points
print(energy.energy)
search_inds = energy.indices 

## Adaptive Sampling Build

In [ ]:
energy = LowRankEnergy(X, p=2)  
adapsampling = AdaptiveAlgorithm(energy, seed=10)
adapsampling.build_phase(5, "sampling")
print(energy.energy)
sampling_inds = energy.indices

## Adaptive Sampling Build + Adaptive Search Swap

In [ ]:
energy = LowRankEnergy(X, p=2)  
sampling_search = AdaptiveAlgorithm(energy, seed=10)
sampling_search.build_phase(5, "sampling")
sampling_search.swap_phase("search")
print(energy.energy)